# Day 6 — Ensemble Scoring Pipeline (SMD Dataset)

Combines Isolation Forest (5% weight) and TCN Autoencoder (95% weight) into a weighted ensemble anomaly detector. Both models were trained on the same 7-machine SMD subset in Days 4 and 5.

**Depends on Day 4 and Day 5 artifacts:**
- `artifacts/isolation_forest.joblib`
- `artifacts/tcn_autoencoder.pt`
- `artifacts/feature_pipeline.joblib`
- `artifacts/day4_if_metrics.json`

In [ ]:
import os
import sys
from pathlib import Path

# Project root resolution — handles VS Code Jupyter working directory
_root = Path.cwd()
while not (_root / "src").exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
os.chdir(_root)

print(f"Project root: {_root}")
print(f"Working dir:  {Path.cwd()}")

In [ ]:
import json
import logging
import numpy as np
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s",
)
logger = logging.getLogger("day6_smd")

ARTIFACTS_DIR = Path("artifacts")
MACHINES = [f"machine-1-{i}" for i in range(1, 8)]
print("Logging configured.")

## Step 1 — Load Day 4/5 Artifacts

Reads `feature_cols` and `input_dim` from Day 4 metrics, loads the fitted normalization pipeline, and confirms all three model artifacts exist before proceeding.

In [ ]:
metrics_path = ARTIFACTS_DIR / "day4_if_metrics.json"
if not metrics_path.exists():
    raise FileNotFoundError(f"Day 4 metrics not found at {metrics_path}. Run Day 4 first.")

with open(metrics_path) as f:
    day4_metrics = json.load(f)

feature_cols = day4_metrics["feature_cols"]
input_dim    = day4_metrics["input_dim_for_tcn"]

for artifact in ["isolation_forest.joblib", "tcn_autoencoder.pt", "feature_pipeline.joblib"]:
    path = ARTIFACTS_DIR / artifact
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}")
    print(f"✓  {artifact}  ({path.stat().st_size / 1024:.1f} KB)")

print(f"\nfeature_cols: {len(feature_cols)} columns")
print(f"input_dim:    {input_dim}")

In [ ]:
from src.features.engineering import load_feature_pipeline

pipeline = load_feature_pipeline()

# Patch NaN bounds (cpu_mem_corr_long rolling Pearson edge case)
normalizer = pipeline.named_steps["normalizer"]
n_patched = 0
for col in list(normalizer.bounds_.keys()):
    lo, hi = normalizer.bounds_[col]
    if np.isnan(lo) or np.isnan(hi) or np.isinf(lo) or np.isinf(hi):
        normalizer.bounds_[col] = (0.0, 1.0)
        logger.warning("Patched NaN bounds for '%s'", col)
        n_patched += 1

print(f"Feature pipeline loaded. NaN bounds patched: {n_patched} columns")

## Step 2 — Recreate Normalized Feature Splits

Identical to Days 4 and 5. The same 7-machine subset, same 70/15/15 split, same fitted normalization pipeline. Takes approximately 40 seconds.

In [ ]:
from src.data.ingestion import load_smd_dataset
from src.data.validation import validate_smd_schema, define_temporal_split_per_series
from src.features.engineering import build_alibaba_features, apply_feature_pipeline

raw_df = load_smd_dataset(machines=MACHINES)
raw_df = validate_smd_schema(raw_df)
feat_df = build_alibaba_features(raw_df, group_col="machine_id")

train_df, val_df, test_df = define_temporal_split_per_series(
    feat_df, group_col="machine_id", train_pct=0.70, val_pct=0.15,
)

train_norm = apply_feature_pipeline(pipeline, train_df, feature_cols)
val_norm   = apply_feature_pipeline(pipeline, val_df,   feature_cols)
test_norm  = apply_feature_pipeline(pipeline, test_df,  feature_cols)

# NaN guard
for label, df in [("train_norm", train_norm), ("val_norm", val_norm), ("test_norm", test_norm)]:
    n_nan = df[feature_cols].isna().sum().sum()
    if n_nan > 0:
        logger.warning("%s: %d NaN values — filling with 0.0", label, n_nan)
        df[feature_cols] = df[feature_cols].fillna(0.0)

print(f"Train: {len(train_norm):,} rows | {train_norm['is_anomaly'].mean()*100:.2f}% anomaly")
print(f"Val:   {len(val_norm):,} rows  | {val_norm['is_anomaly'].mean()*100:.2f}% anomaly")
print(f"Test:  {len(test_norm):,} rows  | {test_norm['is_anomaly'].mean()*100:.2f}% anomaly")

## Step 3 — Run Ensemble Pipeline

Combines IF scores (5%) and TCN reconstruction error scores (95%) into a single weighted ensemble score. Threshold is calibrated on the validation set, then applied to both validation and test sets.

In [ ]:
from src.models.ensemble import run_ensemble_pipeline

logger.info("Running ensemble pipeline (IF=5%%, TCN=95%%)...")

results = run_ensemble_pipeline(
    train_df=train_norm,
    val_df=val_norm,
    test_df=test_norm,
    feature_cols=feature_cols,
)

print("Ensemble pipeline complete.")

## Step 4 — Results

The key number is **test AUC-ROC**. With IF at 0.894 and TCN at 0.887 individually, the ensemble should meet or exceed both components — the two models make errors on different samples, so the weighted combination captures more signal than either alone.

In [ ]:
val_metrics  = results["val_metrics"]
test_metrics = results["test_metrics"]

print("=" * 58)
print("CLOUDDRIFT ENSEMBLE  —  SMD  —  7 MACHINES")
print("=" * 58)

print("\n--- Validation Set ---")
print(f"  Precision:  {val_metrics['precision']:.3f}  (target ≥0.70: {'✓' if val_metrics['precision'] >= 0.70 else '✗'})")
print(f"  Recall:     {val_metrics['recall']:.3f}  (target ≥0.65: {'✓' if val_metrics['recall'] >= 0.65 else '✗'})")
print(f"  F1:         {val_metrics['f1']:.3f}")
print(f"  F2:         {val_metrics['f2']:.3f}")
print(f"  AUC-ROC:    {val_metrics['auc_roc']:.3f}")

print("\n--- Test Set ---")
print(f"  Precision:  {test_metrics['precision']:.3f}  (target ≥0.70: {'✓' if test_metrics['precision'] >= 0.70 else '✗'})")
print(f"  Recall:     {test_metrics['recall']:.3f}  (target ≥0.65: {'✓' if test_metrics['recall'] >= 0.65 else '✗'})")
print(f"  F1:         {test_metrics['f1']:.3f}")
print(f"  F2:         {test_metrics['f2']:.3f}")
print(f"  AUC-ROC:    {test_metrics['auc_roc']:.3f}")

In [ ]:
# Component comparison
day5_path = ARTIFACTS_DIR / "day5_tcn_metrics.json"
if day5_path.exists():
    with open(day5_path) as f:
        day5 = json.load(f)
    print("\n--- Component Comparison (Test AUC-ROC) ---")
    print(f"  Isolation Forest standalone:  {day4_metrics['test_metrics']['auc_roc']:.3f}")
    print(f"  TCN Autoencoder standalone:   {day5['test_metrics']['auc_roc']:.3f}")
    print(f"  Ensemble (IF 5%% + TCN 95%%):  {test_metrics['auc_roc']:.3f}")
    print()

# Top anomalies
if "top_anomalies" in results and results["top_anomalies"] is not None:
    print("--- Top 10 Anomalies by Ensemble Score ---")
    print(results["top_anomalies"].head(10).to_string(index=False))
print("=" * 58)

## Step 5 — Save Ensemble Metrics

In [ ]:
ensemble_out = {
    "dataset": "SMD",
    "machines": MACHINES,
    "n_machines": len(MACHINES),
    "weights": {"isolation_forest": 0.05, "tcn_autoencoder": 0.95},
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "component_test_auc_roc": {
        "isolation_forest": day4_metrics["test_metrics"]["auc_roc"],
        "tcn_autoencoder": day5["test_metrics"]["auc_roc"] if day5_path.exists() else None,
        "ensemble": test_metrics["auc_roc"],
    },
}

ensemble_path = ARTIFACTS_DIR / "day6_ensemble_metrics.json"
with open(ensemble_path, "w") as f:
    json.dump(ensemble_out, f, indent=2, default=str)

print(f"Ensemble metrics saved: {ensemble_path}")
print("Day 6 complete. CloudDrift ensemble pipeline finished.")